# P3 · Modelo predictivo end-to-end
## Dataset: Airbnb Buenos Aires — listings_clean.csv (desde P1)
**Objetivo:** Clasificar propiedades según su frecuencia de arriendo
(alta vs baja) usando modelos de ML clásicos y una red neuronal con Keras.

## Estructura

1. Carga y exploración inicial
2. Feature engineering
3. Preparación del dataset (train/val/test)
4. Modelos clásicos: regresión logística, árboles, XGBoost, LightGBM
5. Evaluación rigurosa: ROC-AUC, F1, matriz de confusión
6. Interpretación: importancia de variables
7. Red neuronal con Keras
8. Comparativa final: clásicos vs Keras

## 1. Carga y exploración inicial (Markdown)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/listings_clean.csv")
df["first_review"] = pd.to_datetime(df["first_review"])
df["last_review"] = pd.to_datetime(df["last_review"])
df["last_scraped"] = pd.to_datetime(df["last_scraped"])

print(df.shape)

(27348, 72)


## Variable objetivo

In [6]:
# Creamos la variable objetivo: alta_frecuencia
# 1 si reviews_per_month está sobre la mediana, 0 si está bajo
# Solo para propiedades con reseñas

df_modelo = df[df["tiene_resenas"] == 1].copy()

mediana_reviews = df_modelo["reviews_per_month"].median()
df_modelo["alta_frecuencia"] = (df_modelo["reviews_per_month"] > mediana_reviews).astype(int)

print(f"Mediana de reviews_per_month: {mediana_reviews:.2f}")
print(f"\nDistribución de la variable Objetivo:")
print(df_modelo["alta_frecuencia"].value_counts())
print(f"\nBalance: {df_modelo["alta_frecuencia"].mean():.2%}")

Mediana de reviews_per_month: 1.00

Distribución de la variable Objetivo:
alta_frecuencia
0    12282
1    11763
Name: count, dtype: int64

Balance: 48.92%


## 2. Feature engineering

In [8]:
features = ["room_type", "bathrooms", "beds", "review_scores_rating",
            "neighbourhood_cleansed", "antiguedad_total", "accommodates"]

df_modelo_feature = df_modelo[features + ["alta_frecuencia"]].dropna()

print(f"Shape: {df_modelo_feature.shape}")
print(f"Balance: {df_modelo_feature['alta_frecuencia'].mean():.2%}")

Shape: (24045, 8)
Balance: 48.92%


In [9]:
# One-hot Encoding para variables categóricas

df_encoded = pd.get_dummies(
    df_modelo_feature,
    columns = ["room_type", "neighbourhood_cleansed"],
    drop_first = True
)

print(f"Shape antes: {df_modelo_feature.shape}")
print(f"Shape después: {df_encoded.shape}")
print(f"Columnas nuevas: {df_encoded.shape[1] - df_modelo_feature.shape[1]}")

Shape antes: (24045, 8)
Shape después: (24045, 55)
Columnas nuevas: 47


## 3. Preparación del dataset (train/val/test)

In [11]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["alta_frecuencia"])
y = df_encoded["alta_frecuencia"]

# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size = 0.30, random_state = 42, stratify = y
    )
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size = 0.50, random_state = 42, stratify = y_temp)

print(f"Train: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X):.0%})")
print(f"Val:   {X_val.shape[0]:,} filas ({X_val.shape[0]/len(X):.0%})")
print(f"Test:  {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X):.0%})")


Train: 16,831 filas (70%)
Val:   3,607 filas (15%)
Test:  3,607 filas (15%)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 4. Modelos clásicos

### Regresión Logística

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# Baseline: regresión logística
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# Evaluación en validación
y_pred_lr = lr.predict(X_val_scaled)      
y_prob_lr = lr.predict_proba(X_val_scaled)[:, 1] 

print("=== Regresión Logística ===")
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_lr):.4f}")
print(f"F1:      {f1_score(y_val, y_pred_lr):.4f}")
print(f"\n{classification_report(y_val, y_pred_lr)}")

=== Regresión Logística ===
ROC-AUC: 0.6600
F1:      0.6405

              precision    recall  f1-score   support

           0       0.65      0.57      0.61      1843
           1       0.60      0.68      0.64      1764

    accuracy                           0.62      3607
   macro avg       0.63      0.63      0.62      3607
weighted avg       0.63      0.62      0.62      3607



### XGBoost

In [16]:
from xgboost import XGBClassifier

xgb = XGBClassifier (random_state=42, eval_metric="logloss")
xgb.fit(X_train_scaled, y_train)

y_pred_xgb = xgb.predict(X_val_scaled)
y_prob_xgb = xgb.predict_proba(X_val_scaled)[:,1]

print("=== XGBoost ===")
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_xgb):.4f}")
print(f"F1:      {f1_score(y_val, y_pred_xgb):.4f}")
print(f"\n{classification_report(y_val, y_pred_xgb)}")

=== XGBoost ===
ROC-AUC: 0.8277
F1:      0.7372

              precision    recall  f1-score   support

           0       0.75      0.72      0.74      1843
           1       0.72      0.76      0.74      1764

    accuracy                           0.74      3607
   macro avg       0.74      0.74      0.74      3607
weighted avg       0.74      0.74      0.74      3607



### LightGBM

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier (random_state=42, eval_metric="logloss")
lgbm.fit(X_train_scaled, y_train)

y_pred_lgbm = lgbm.predict(X_val_scaled)
y_prob_lgbm = lgbm.predict_proba(X_val_scaled)[:, 1]

print("=== LightGBM ===")
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_xgb):.4f}")
print(f"F1:      {f1_score(y_val, y_pred_xgb):.4f}")
print(f"\n{classification_report(y_val, y_pred_xgb)}")